In [10]:
%pip install markdown

Note: you may need to restart the kernel to use updated packages.


In [1]:
# Load env variables and create client
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()

client = Anthropic()
model = "claude-sonnet-4-5"

In [2]:
# Helper functions
from anthropic.types import Message


def add_user_message(messages, message):
    user_message = {
        "role": "user",
        "content": message.content if isinstance(message, Message) else message,
    }
    messages.append(user_message)


def add_assistant_message(messages, message):
    assistant_message = {
        "role": "assistant",
        "content": message.content if isinstance(message, Message) else message,
    }
    messages.append(assistant_message)


def chat(messages, system=None, temperature=1.0, stop_sequences=[], tools=None):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences,
    }

    if tools:
        params["tools"] = tools

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message


def text_from_message(message):
    return "\n".join([block.text for block in message.content if block.type == "text"])

In [3]:
web_search_schema = {
    "type": "web_search_20250305",
    "name": "web_search",
    "max_uses": 5,
    "allowed_domains": ["nih.gov"],
}

In [13]:
messages = []
add_user_message(
    messages,
    """
    What could be the reason of rear deltoid pain? Search the web and cite sources."
    """,
)
response = chat(messages, tools=[web_search_schema])

In [15]:
from urllib.parse import urlparse
from markdown import markdown


def render_message_html(response, output_file="web_search_tool_result.html"):
    """
    Render Message object to HTML:
    - Sources at top
    - Markdown converted to HTML
    - Inline citations
    """

    html = """
    <html>
    <head>
        <meta charset="utf-8">
        <title>Chat Response</title>
        <style>
            body {
                font-family: Arial, sans-serif;
                max-width: 900px;
                margin: 40px auto;
                padding: 20px;
                line-height: 1.6;
            }

            .sources {
                background: #f5f5f5;
                padding: 16px;
                border-radius: 10px;
                margin-bottom: 30px;
            }

            .source {
                margin-bottom: 14px;
            }

            .citation {
                background: #fafafa;
                border-left: 4px solid #ccc;
                padding: 10px;
                margin: 12px 0;
            }

            code {
                background: #eee;
                padding: 2px 5px;
                border-radius: 4px;
            }
        </style>
    </head>
    <body>
    """

    # -------------------------
    # Collect all sources
    # -------------------------
    all_sources = {}

    for block in response.content:
        if getattr(block, "citations", None):
            for citation in block.citations:

                url = citation.url

                all_sources[url] = {
                    "title": getattr(citation, "title", "Untitled"),
                    "url": url,
                    "domain": urlparse(url).netloc,
                }

    # -------------------------
    # Sources section
    # -------------------------
    if all_sources:

        html += "<div class='sources'>"
        html += "<h2>Sources</h2>"

        for source in all_sources.values():

            html += f"""
            <div class="source">
                <strong>{source['title']}</strong><br>
                Domain: {source['domain']}<br>
                <a href="{source['url']}" target="_blank">
                    {source['url']}
                </a>
            </div>
            """

        html += "</div>"

    # -------------------------
    # Content section
    # -------------------------
    for block in response.content:

        if block.type != "text":
            continue

        # Convert markdown -> HTML
        html += markdown(block.text)

        # Inline citations
        if getattr(block, "citations", None):

            html += "<h3>Citations</h3>"

            for citation in block.citations:

                url = citation.url
                title = getattr(citation, "title", "Untitled")
                domain = urlparse(url).netloc
                quote = getattr(citation, "quote", "")

                html += f"""
                <div class="citation">
                    <strong>{title}</strong><br>
                    Domain: {domain}<br>
                    URL:
                    <a href="{url}" target="_blank">{url}</a><br><br>

                    <em>"{quote}"</em>
                </div>
                """

    html += "</body></html>"

    # Save HTML file
    with open(output_file, "w", encoding="utf-8") as f:
        f.write(html)

    print(f"Saved HTML to: {output_file}")

render_message_html(response)

Saved HTML to: web_search_tool_result.html
